# Task 2 — Technical Analysis

**Objective:** Compute common technical indicators (SMA, EMA, RSI, MACD) from stock price data and create publication-quality overlay charts.

**Learning goals:**
- Implement moving averages, RSI, and MACD from scratch
- Interpret indicator signals in a financial context
- Create clean, publication-ready subplots

## 1. Setup

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.indicators import sma, ema, rsi, macd, daily_returns
from src.data_loader import fetch_stock_prices
from src.visualization import time_series

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

## 2. Load Price Data

In [ ]:
prices_path = root / "data" / "raw" / "stock_prices.csv"

if prices_path.exists():
    prices = pd.read_csv(prices_path, index_col=0, parse_dates=True)
else:
    prices = fetch_stock_prices(["AAPL", "GOOG", "META"], start="2023-01-01")

print(f"Loaded {prices.shape[0]} rows x {prices.shape[1]} columns")
prices.tail()

## 3. Moving Averages (SMA & EMA)

We pick one ticker (AAPL) for a detailed visual walkthrough.

In [ ]:
ticker = "AAPL"
close = prices[ticker].dropna()

# Compute indicators
close_sma20 = sma(close, window=20)
close_sma50 = sma(close, window=50)
close_ema20 = ema(close, span=20)

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(close.index, close, label="Close", linewidth=1.5, color="black")
ax.plot(close.index, close_sma20, label="SMA(20)", linewidth=1.2)
ax.plot(close.index, close_sma50, label="SMA(50)", linewidth=1.2, linestyle="--")
ax.plot(close.index, close_ema20, label="EMA(20)", linewidth=1.2, linestyle=":")

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()

ax.set(title=f"{ticker} — Moving Averages", ylabel="Price (USD)", xlabel="")
ax.legend()
sns.despine()
plt.show()

## 4. Relative Strength Index (RSI)

In [ ]:
close_rsi = rsi(close, window=14)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price panel
ax1.plot(close.index, close, color="black", linewidth=1.2)
ax1.set(ylabel="Price (USD)", title=f"{ticker} — Price and RSI(14)")

# RSI panel
ax2.plot(close_rsi.index, close_rsi, color="purple", linewidth=1.2)
ax2.axhline(70, color="red", linestyle="--", alpha=0.5, label="Overbought (70)")
ax2.axhline(30, color="green", linestyle="--", alpha=0.5, label="Oversold (30)")
ax2.fill_between(close_rsi.index, 70, 100, alpha=0.08, color="red")
ax2.fill_between(close_rsi.index, 0, 30, alpha=0.08, color="green")
ax2.set(ylabel="RSI", ylim=(0, 100))
ax2.legend(loc="upper right")

ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()
sns.despine()
plt.show()

**Interpretation:** RSI above 70 suggests overbought conditions, while RSI below 30 suggests oversold. Crossovers of these thresholds can signal potential reversals.

## 5. MACD (Moving Average Convergence Divergence)

In [ ]:
macd_df = macd(close, fast=12, slow=26, signal=9)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price panel
ax1.plot(close.index, close, color="black", linewidth=1.2)
ax1.set(ylabel="Price (USD)", title=f"{ticker} — Price and MACD")

# MACD panel
ax2.plot(macd_df.index, macd_df["macd"], label="MACD", color="blue", linewidth=1.2)
ax2.plot(macd_df.index, macd_df["signal"], label="Signal", color="orange", linewidth=1.2)
ax2.bar(
    macd_df.index,
    macd_df["histogram"],
    width=1.5,
    color=np.where(macd_df["histogram"] >= 0, "green", "red"),
    alpha=0.4,
    label="Histogram",
)
ax2.axhline(0, color="gray", linewidth=0.5)
ax2.set(ylabel="MACD", xlabel="")
ax2.legend(loc="upper left")

ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate()
sns.despine()
plt.show()

**Interpretation:** When the MACD line crosses above the signal line it is a bullish signal; a cross below is bearish. The histogram shows the momentum strength.

## 6. Summary

- We implemented SMA, EMA, RSI, and MACD using only `pandas` vectorised operations.
- Visual overlays make it easy to spot crossovers and extreme RSI regions.
- These indicators will serve as features in the sentiment correlation analysis (Task 3).